# 05 — Kurgu A üstüne origin ekleri

Bu notebook sub_04 / Kurgu A tarifinin üzerine kurulur. Cold-start, hava durumu ve yilin_gunu yok.

Kurgu A (Kaggle 1.06168):
- özellik: 24 sütun, yilin_gunu yok
- num_leaves=31, learning_rate=0.05, 87 ağaç
- origin: Nisan-Temmuz, Mayıs-Ağustos, Haziran-Eylül, Aralık-Mart

sub_02 (Kaggle 1.08157) aynı yaz originlerine ek olarak Ocak ve Şubat
origin'leri de kullanıyordu ve 150 ağaçla eğitilmişti. Daha kötü skorun
sebebi büyük ihtimalle yilin_gunu ve leaves=63. Veri eksiği değil.

Burada Kurgu A sabit, sadece eğitim seti büyütüyoruz

1. sub_09: Kurgu A originleri + Ocak + Şubat (tek fark: eksik kış)
2. sub_10: sub_02'nin origin karışımı, Kurgu A özellik/ağaç tarifi


In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb

train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih']  = pd.to_datetime(test['tarih'])

# 444 satır güç x 24 saat sınırını aşıyor. Trafo medyanı ile düzeltiyoruz.
train['asiri_oran'] = train['tuketim'] / (train['guc'] * 24)
train['tuketim_duzeltilmis'] = train['tuketim']
for t in train.loc[train['asiri_oran'] > 2, 'tanim'].unique():
    medyan = train.loc[(train['tanim'] == t) & (train['asiri_oran'] <= 2), 'tuketim'].median()
    train.loc[(train['tanim'] == t) & (train['asiri_oran'] > 2), 'tuketim_duzeltilmis'] = medyan

TATILLER = pd.to_datetime([
    '2025-01-01','2025-03-29','2025-03-30','2025-03-31','2025-04-01','2025-04-23',
    '2025-05-01','2025-05-19','2025-06-05','2025-06-06','2025-06-07','2025-06-08','2025-06-09',
    '2025-08-30','2025-10-29',
    '2026-01-01','2026-03-19','2026-03-20','2026-03-21','2026-04-23','2026-05-01','2026-05-19',
    '2026-05-26','2026-05-27','2026-05-28','2026-05-29','2026-05-30','2026-08-30','2026-10-29',
]).date

GUC_BINS = [0, 100, 250, 500, 1000, 2000, 5000, np.inf]


def temel_ozellikler(df):
    df = df.copy()
    lok = df['lokasyon'].str.split('>', expand=True)
    df['il']    = lok[0]
    df['bolge'] = lok[1]
    df['ilce']  = (lok[2] if lok.shape[1] > 2 else pd.Series(np.nan, index=df.index)).fillna('YOK')
    df['ay']            = df['tarih'].dt.month
    df['gun']           = df['tarih'].dt.day
    df['haftanin_gunu'] = df['tarih'].dt.dayofweek
    df['hafta_sonu']    = (df['haftanin_gunu'] >= 5).astype(int)
    df['tatil']         = df['tarih'].dt.date.isin(TATILLER).astype(int)
    df['guc_bucket']     = pd.cut(df['guc'], bins=GUC_BINS)
    df['guc_bucket_str'] = df['guc_bucket'].astype(str)
    return df


train = temel_ozellikler(train)
test  = temel_ozellikler(test)
KATEGORIK = ['il', 'bolge', 'ilce', 'guc_bucket_str']

print('train', train.shape, '| test', test.shape)
print('düzeltilen satır:', (train['tuketim'] != train['tuketim_duzeltilmis']).sum())


train (1226237, 17) | test (714688, 15)
düzeltilen satır: 444


In [4]:
def trafo_profili(gecmis, origin):
    g = gecmis[gecmis['tarih'] < origin].copy()
    g['log_t'] = np.log1p(g['tuketim_duzeltilmis'])
    profil = g.groupby('tanim').agg(
        trafo_ortalama   = ('tuketim_duzeltilmis', 'mean'),
        trafo_medyan     = ('tuketim_duzeltilmis', 'median'),
        trafo_gun_sayisi = ('tuketim_duzeltilmis', 'count'),
        log_ortalama     = ('log_t', 'mean'),
        log_medyan       = ('log_t', 'median'),
        log_std          = ('log_t', 'std'),
        sifir_orani      = ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
    ).reset_index()
    for gun in [30, 90]:
        son = g[g['tarih'] >= origin - pd.Timedelta(days=gun)]
        ozet = son.groupby('tanim').agg(**{
            f'son{gun}_log_ort': ('log_t', 'mean'),
            f'son{gun}_sifir'  : ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
        }).reset_index()
        profil = profil.merge(ozet, on='tanim', how='left')
    profil['log_trend'] = profil['son30_log_ort'] - profil['son90_log_ort']
    return profil


def trafo_gun_profili(gecmis, origin):
    g = gecmis[gecmis['tarih'] < origin][['tanim', 'haftanin_gunu', 'tuketim_duzeltilmis']].copy()
    ort = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_ort')
    g = g.merge(ort, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_ort'].replace(0, np.nan)
    return g.groupby(['tanim', 'haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()


def veri_seti_kur(gecmis, hedef, origin, k=30):
    g = gecmis[gecmis['tarih'] < origin].copy()
    g['log_t'] = np.log1p(g['tuketim_duzeltilmis'])
    genel = g['log_t'].mean()

    grup = g.groupby(['il', 'bolge', 'guc_bucket'], observed=True)['log_t'].agg(['sum', 'count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k * genel) / (grup['count'] + k)
    il_guc = g.groupby(['il', 'guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index()

    df = hedef.copy()
    df = df.merge(grup[['il', 'bolge', 'guc_bucket', 'grup_log_ort']],
                  on=['il', 'bolge', 'guc_bucket'], how='left')
    df = df.merge(il_guc, on=['il', 'guc_bucket'], how='left')
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel)
    df = df.merge(trafo_profili(gecmis, origin), on='tanim', how='left')
    df = df.merge(trafo_gun_profili(gecmis, origin), on=['tanim', 'haftanin_gunu'], how='left')
    return df


def origin_seti(bas, bit, origin):
    hedef = train[(train['tarih'] >= bas) & (train['tarih'] <= bit)]
    return veri_seti_kur(train, hedef, pd.Timestamp(origin))


# Kurgu A özellik seti: yilin_gunu yok, hava yok
OZELLIKLER = [
    'guc', 'ay', 'gun', 'haftanin_gunu', 'hafta_sonu', 'tatil',
    'il', 'bolge', 'ilce', 'guc_bucket_str', 'grup_log_ort',
    'trafo_ortalama', 'trafo_medyan', 'trafo_gun_sayisi',
    'log_ortalama', 'log_medyan', 'log_std', 'sifir_orani',
    'son30_log_ort', 'son90_log_ort', 'son30_sifir', 'son90_sifir',
    'log_trend', 'trafo_gun_orani',
]
print('özellik sayısı:', len(OZELLIKLER), '(Kurgu A / sub_04 ile aynı)')


özellik sayısı: 24 (Kurgu A / sub_04 ile aynı)


In [5]:
KURGU_A = [
    ('2025-04-01', '2025-07-31'),
    ('2025-05-01', '2025-08-31'),
    ('2025-06-01', '2025-09-30'),
    ('2025-12-01', '2026-03-31'),
]

# sub_09: Kurgu A + sub_02'de olup burada eksik olan kış origin'leri
KURGU_A_KIS = KURGU_A + [
    ('2026-01-01', '2026-03-31'),
    ('2026-02-01', '2026-03-31'),
]

# sub_10: sub_02'nin origin karışımı (yaz hedefleri Eylül'e kadar)
SUB02_ORIGIN = [
    ('2025-04-01', '2025-09-30'),
    ('2025-05-01', '2025-09-30'),
    ('2025-06-01', '2025-09-30'),
    ('2025-12-01', '2026-03-31'),
    ('2026-01-01', '2026-03-31'),
    ('2026-02-01', '2026-03-31'),
]


def set_kur(originler):
    df = pd.concat([origin_seti(b, s, b) for b, s in originler], ignore_index=True)
    df['hedef_log'] = np.log1p(df['tuketim_duzeltilmis'])
    for c in KATEGORIK:
        df[c] = df[c].astype('category')
    return df


def ozet(ad, df):
    print(f'{ad:22s} satır {len(df):,}  |  aylar {sorted(df["ay"].unique())}')



egitim_a   = set_kur(KURGU_A)
egitim_kis = set_kur(KURGU_A_KIS)
egitim_02  = set_kur(SUB02_ORIGIN)

ozet('Kurgu A (referans)', egitim_a)
ozet('A + Oca/Şub', egitim_kis)
ozet('sub_02 origin mix', egitim_02)
print('test ayları:', sorted(test['ay'].unique()))


Kurgu A (referans)     satır 1,306,364  |  aylar [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(12)]
A + Oca/Şub            satır 1,870,192  |  aylar [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(12)]
sub_02 origin mix      satır 2,096,870  |  aylar [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(12)]
test ayları: [np.int32(4), np.int32(5), np.int32(6), np.int32(7)]


In [ ]:
AGAC = 87 # Kurgu A'daki ağaç sayısı, origin farkını izole etmek için sabit
LEAVES = 31
LR = 0.05


def egit(data):
    model = lgb.LGBMRegressor(
        n_estimators=AGAC, learning_rate=LR, num_leaves=LEAVES,
        random_state=42, verbose=-1,
    )
    model.fit(data[OZELLIKLER], data['hedef_log'], categorical_feature=KATEGORIK)
    return model


print(f'Kurgu A tarifi: leaves={LEAVES}, lr={LR}, ağaç={AGAC}')
model_kis = egit(egitim_kis)
model_02  = egit(egitim_02)



Kurgu A tarifi: leaves=31, lr=0.05, ağaç=87

A + Oca/Şub eğitiliyor...
sub_02 origin mix eğitiliyor...
tamam


In [7]:
test_seti = veri_seti_kur(train, test, pd.Timestamp('2026-04-01'))
sample = pd.read_csv('../data/raw/sample_submission.csv')


def kaydet(model, egitim, dosya):
    ts = test_seti.copy()
    for c in KATEGORIK:
        ts[c] = pd.Categorical(ts[c], categories=egitim[c].cat.categories)
    tahmin = np.clip(np.expm1(model.predict(ts[OZELLIKLER])), 0, None)
    sub = pd.DataFrame({'id': ts['id'], 'tuketim': tahmin})
    ok = (len(sub) == len(sample)
          and set(sub['id']) == set(sample['id'])
          and sub['tuketim'].isna().sum() == 0
          and (sub['tuketim'] < 0).sum() == 0)
    sub.to_csv(f'../submissions/{dosya}.csv', index=False)
    print(f'{dosya:32s} ortalama {sub["tuketim"].mean():7.1f}  '
          f'medyan {sub["tuketim"].median():7.1f}  max {sub["tuketim"].max():9.1f}  '
          f'| doğrulama {"tamam" if ok else "HATA"}')
    return sub


sub_09 = kaydet(model_kis, egitim_kis, 'sub_09_kurguA_kis_origin')
sub_10 = kaydet(model_02,  egitim_02,  'sub_10_kurguA_sub02_origin')

taban = pd.read_csv('../submissions/sub_04_kurguA_sade.csv')
print()
print('sub_04 ortalama / medyan:', round(taban.tuketim.mean(), 1), '/', round(taban.tuketim.median(), 1))


sub_09_kurguA_kis_origin         ortalama  1856.3  medyan  1235.7  max   87114.3  | doğrulama tamam
sub_10_kurguA_sub02_origin       ortalama  1874.8  medyan  1235.7  max   89221.7  | doğrulama tamam

sub_04 ortalama / medyan: 1890.0 / 1237.0
